# Graph Refinement through Entity Canonicalisation

## Aim of this Notebook
Notebook 3 showed the mention-level knowledge graph is extremely fragmented (11,672
weakly connected components across 12,679 nodes). This notebook merges entity
mentions that refer to the same real-world entity, through four progressively
stronger criteria (V1–V4), producing the canonical graph used from Notebook 5 onward.

**Scope note:** V1–V3 are text-normalisation based (case, whitespace, punctuation).
V4 adds similarity-based fuzzy matching to catch near-duplicate spellings that exact
normalisation cannot. Limitations of this approach are discussed in the conclusion.

### Setup

In [1]:
import json
import pickle
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import networkx as nx

DATA_DIR = Path("../data/archive")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_PATH = DATA_DIR / "all.jsonl"

print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_DIR)

Dataset path: ..\data\archive\all.jsonl
Output directory: ..\outputs


### Load dataset

In [2]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            records.append(json.loads(line))
    return pd.DataFrame(records)

df_all = load_jsonl(DATASET_PATH)

print("Total reports:", len(df_all))
df_all.head()

Total reports: 9732


,id,text,entities,relations,Comments
0,1,This post is also available in: 日本語 (Japa...,"[{'id': 45800, 'label': 'malware', 'start_offs...",[],[]
1,2,The attack vector is very basic and repeats it...,"[{'id': 48941, 'label': 'attack-pattern', 'sta...",[],[]
2,3,Once executed by the user the first stage malw...,[],[],[]
3,4,The first known campaign was launched by Crim...,"[{'id': 45806, 'label': 'TIME', 'start_offset'...",[],[]
4,5,The first stage downloaded the ransomware from...,[],[],[]


### Build entity table with actual entity text

In [3]:
entity_records = []

for _, row in df_all.iterrows():
    report_id = row["id"]
    text = row["text"]

    for entity in row["entities"]:
        start = entity.get("start_offset")
        end = entity.get("end_offset")

        entity_text = None
        if pd.notna(start) and pd.notna(end):
            entity_text = text[int(start):int(end)]

        entity_records.append({
            "entity_id": entity.get("id"),
            "entity_type": entity.get("label"),
            "entity_text": entity_text,
            "start_offset": start,
            "end_offset": end,
            "report_id": report_id
        })

entity_df = pd.DataFrame(entity_records)

print("Total entity records:", len(entity_df))
print("Unique original entity IDs:", entity_df["entity_id"].nunique())

entity_df.head()

Total entity records: 12679
Unique original entity IDs: 12679


,entity_id,entity_type,entity_text,start_offset,end_offset,report_id
0,45800,malware,CryptoLocker,288,300,1
1,45801,malware,CTB-Locker,53,63,1
2,45802,malware,CTB-Locker,342,352,1
3,48941,attack-pattern,spear phishing email sent with SCR attachments,69,115,2
4,45806,TIME,November 2014,55,68,4


### Build relation table

In [4]:
relation_records = []

for _, row in df_all.iterrows():
    report_id = row["id"]

    for relation in row["relations"]:
        relation_records.append({
            "relation_id": relation.get("id"),
            "source_id": relation.get("from_id"),
            "target_id": relation.get("to_id"),
            "relation_type": relation.get("type"),
            "report_id": report_id
        })

relation_df = pd.DataFrame(relation_records)

print("Total relation records:", len(relation_df))
relation_df.head()

Total relation records: 1095


,relation_id,source_id,target_id,relation_type,report_id
0,1,44599,44600,uses,105
1,2,44600,44601,exfiltrates-to,105
2,3,44599,44601,exfiltrates-to,105
3,120,45834,45835,uses,106
4,121,45834,45836,uses,106


### Criteria 1: Exact Entity Matching Using Text Normalisation

In [5]:
def normalise_entity_text(text):
    if pd.isna(text):
        return None

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("_", "-")

    return text

entity_df["normalised_text"] = entity_df["entity_text"].apply(normalise_entity_text)

entity_df["canonical_key"] = (
    entity_df["entity_type"].astype(str)
    + "::"
    + entity_df["normalised_text"].astype(str)
)

entity_df.head()

,entity_id,entity_type,entity_text,start_offset,end_offset,report_id,normalised_text,canonical_key
0,45800,malware,CryptoLocker,288,300,1,cryptolocker,malware::cryptolocker
1,45801,malware,CTB-Locker,53,63,1,ctb-locker,malware::ctb-locker
2,45802,malware,CTB-Locker,342,352,1,ctb-locker,malware::ctb-locker
3,48941,attack-pattern,spear phishing email sent with SCR attachments,69,115,2,spear phishing email sent with scr attachments,attack-pattern::spear phishing email sent with...
4,45806,TIME,November 2014,55,68,4,november 2014,TIME::november 2014


### Evaluate Criteria 1

In [6]:
# SECTION: EVALUATE CRITERIA 1
canonical_summary = pd.DataFrame([
    ["Original entity records", len(entity_df)],
    ["Unique original entity IDs", entity_df["entity_id"].nunique()],
    ["Unique canonical entities (V1)", entity_df["canonical_key"].nunique()],
    ["Entities reduced by", entity_df["entity_id"].nunique() - entity_df["canonical_key"].nunique()]
], columns=["Metric", "Value"])
display(canonical_summary)

,Metric,Value
0,Original entity records,12679
1,Unique original entity IDs,12679
2,Unique canonical entities (V1),4385
3,Entities reduced by,8294


### Create mapping from old entity ID to canonical key

In [7]:
id_to_canonical = dict(
    zip(entity_df["entity_id"], entity_df["canonical_key"])
)

relation_df["canonical_source"] = relation_df["source_id"].map(id_to_canonical)
relation_df["canonical_target"] = relation_df["target_id"].map(id_to_canonical)

canonical_valid_relations = relation_df[
    relation_df["canonical_source"].notna() &
    relation_df["canonical_target"].notna()].copy()

print("Original relations:", len(relation_df))
print("Canonical valid relations:", len(canonical_valid_relations))

canonical_valid_relations.head()

Original relations: 1095
Canonical valid relations: 1095


,relation_id,source_id,target_id,relation_type,report_id,canonical_source,canonical_target
0,1,44599,44600,uses,105,threat-actor::conti operators,tools::rclone
1,2,44600,44601,exfiltrates-to,105,tools::rclone,SOFTWARE::mega cloud storage service
2,3,44599,44601,exfiltrates-to,105,threat-actor::conti operators,SOFTWARE::mega cloud storage service
3,120,45834,45835,uses,106,threat-actor::darkside operators,tools::mega client
4,121,45834,45836,uses,106,threat-actor::darkside operators,tools::7-zip


###  Relation mapping

In [8]:
merge_examples = (
    entity_df
    .groupby("canonical_key")
    .agg(
        entity_type=("entity_type", "first"),
        canonical_text=("normalised_text", "first"),
        original_texts=("entity_text", lambda x: list(set(x.dropna()))),
        original_entity_ids=("entity_id", list),
        mention_count=("entity_id", "count")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

merged_only = merge_examples[merge_examples["mention_count"] > 1]

print("Number of canonical entities with multiple mentions:", len(merged_only))
display(merged_only.head(20))

Number of canonical entities with multiple mentions: 1263


,canonical_key,entity_type,canonical_text,original_texts,original_entity_ids,mention_count
3147,location::and,location,and,"[and, And]","[6137, 6138, 17215, 17234, 17269, 17280, 17310...",945
2485,attack-pattern::phishing,attack-pattern,phishing,"[phishing , phishing, Phishing]","[715, 45915, 737, 741, 747, 751, 756, 765, 771...",375
3151,location::are,location,are,"[Are, are]","[17191, 17396, 17551, 17612, 17689, 17694, 176...",122
3405,location::security,location,security,"[Security, security]","[17143, 17423, 18324, 18467, 18557, 18575, 186...",121
2954,identity::palo alto networks,identity,palo alto networks,"[ Palo Alto Networks, Palo Alto Networks, Palo...","[22, 31, 45809, 45886, 44649, 45933, 45949, 45...",119
3525,malware::biopass rat,malware,biopass rat,[BIOPASS RAT],"[48292, 48295, 48296, 48300, 48301, 48303, 483...",116
2904,identity::microsoft,identity,microsoft,"[Microsoft , microsoft, Microsoft]","[45850, 44609, 46246, 46247, 44822, 2292, 4633...",99
1358,SOFTWARE::android,SOFTWARE,android,"[Android, Android ]","[46129, 44751, 46131, 45791, 45792, 44757, 447...",99
3182,location::can,location,can,"[Can, can]","[2236, 17232, 17392, 17520, 17523, 17576, 1768...",89
3074,identity::unit 42,identity,unit 42,"[ Unit 42, Unit 42]","[45810, 45840, 45846, 45844, 45891, 45785, 446...",78


### Build Canonical Graph V1

In [9]:
# Build canonical graph using Criteria 1

G_canonical = nx.DiGraph()

canonical_entity_df = (
    entity_df
    .groupby("canonical_key")
    .agg(
        entity_type=("entity_type", "first"),
        canonical_text=("normalised_text", "first"),
        mention_count=("entity_id", "count"),
        original_entity_ids=("entity_id", list),
        report_ids=("report_id", lambda x: list(set(x)))
    )
    .reset_index()
)

for _, row in canonical_entity_df.iterrows():
    G_canonical.add_node(
        row["canonical_key"],
        entity_type=row["entity_type"],
        canonical_text=row["canonical_text"],
        mention_count=row["mention_count"],
        original_entity_ids=row["original_entity_ids"],
        report_ids=row["report_ids"]
    )

for _, row in canonical_valid_relations.iterrows():
    source = row["canonical_source"]
    target = row["canonical_target"]

    if source != target:
        G_canonical.add_edge(
            source,
            target,
            relation_type=row["relation_type"],
            relation_id=row["relation_id"],
            report_id=row["report_id"]
        )

print("Canonical graph created")
print("Nodes:", G_canonical.number_of_nodes())
print("Edges:", G_canonical.number_of_edges())
print("Density:", nx.density(G_canonical))
print("Weakly connected components:", nx.number_weakly_connected_components(G_canonical))

Canonical graph created
Nodes: 4385
Edges: 945
Density: 4.9157712506970515e-05
Weakly connected components: 3571


### Compare original vs canonical graph

In [10]:
# Compare original graph and canonical graph

comparison_df = pd.DataFrame([
    {
        "graph_version": "Original mention-level graph",
        "nodes": 12679,
        "edges": 1094,
        "density": 0.000007,
        "weakly_connected_components": 11672
    },
    {
        "graph_version": "Canonical graph - Criteria 1",
        "nodes": G_canonical.number_of_nodes(),
        "edges": G_canonical.number_of_edges(),
        "density": nx.density(G_canonical),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical)
    }
])

display(comparison_df)

,graph_version,nodes,edges,density,weakly_connected_components
0,Original mention-level graph,12679,1094,0.000007,11672
1,Canonical graph - Criteria 1,4385,945,0.000049,3571


### Criteria 2 — Remove Noise
The noise-term list below was built manually by inspecting frequent low-information
tokens in `entity_text` (stopwords, generic nouns like "tool"/"target"/"source").
It is a heuristic filter, not a learned one, and is not claimed to be exhaustive.


In [11]:
# CRITERIA 2: Remove noisy / generic entity texts before canonical graph construction

noise_terms = {
    "and", "or", "are", "can", "will", "just", "the", "a", "an",
    "to", "of", "in", "on", "for", "with", "by", "from",
    "this", "that", "these", "those", "it", "its", "as", "is",
    "was", "were", "be", "been", "being", "tool", "target",
    "source", "sample", "campaign", "security"
}

entity_df_clean = entity_df[
    entity_df["normalised_text"].notna() &
    ~entity_df["normalised_text"].isin(noise_terms) &
    (entity_df["normalised_text"].str.len() > 2)
].copy()

print("Original entity records:", len(entity_df))
print("After noise filtering:", len(entity_df_clean))
print("Removed noisy records:", len(entity_df) - len(entity_df_clean))
print("Unique canonical entities after Criteria 2:", entity_df_clean["canonical_key"].nunique())

Original entity records: 12679
After noise filtering: 10974
Removed noisy records: 1705
Unique canonical entities after Criteria 2: 4363


###  Update Relation Mapping

In [12]:
# Re-map relations after Criteria 2

id_to_canonical_clean = dict(
    zip(entity_df_clean["entity_id"], entity_df_clean["canonical_key"])
)

relation_df["canonical_source_clean"] = relation_df["source_id"].map(id_to_canonical_clean)
relation_df["canonical_target_clean"] = relation_df["target_id"].map(id_to_canonical_clean)

canonical_valid_relations_clean = relation_df[
    relation_df["canonical_source_clean"].notna() &
    relation_df["canonical_target_clean"].notna()
].copy()

print("Original relations:", len(relation_df))
print("Valid relations after Criteria 2:", len(canonical_valid_relations_clean))

Original relations: 1095
Valid relations after Criteria 2: 1085


### Build Canonical Graph V2

In [13]:
# Build canonical graph after Criteria 2

G_canonical_v2 = nx.DiGraph()

canonical_entity_df_v2 = (
    entity_df_clean
    .groupby("canonical_key")
    .agg(
        entity_type=("entity_type", "first"),
        canonical_text=("normalised_text", "first"),
        mention_count=("entity_id", "count"),
        original_entity_ids=("entity_id", list),
        report_ids=("report_id", lambda x: list(set(x)))
    )
    .reset_index()
)

for _, row in canonical_entity_df_v2.iterrows():
    G_canonical_v2.add_node(
        row["canonical_key"],
        entity_type=row["entity_type"],
        canonical_text=row["canonical_text"],
        mention_count=row["mention_count"]
    )

for _, row in canonical_valid_relations_clean.iterrows():
    source = row["canonical_source_clean"]
    target = row["canonical_target_clean"]

    if source != target:
        G_canonical_v2.add_edge(
            source,
            target,
            relation_type=row["relation_type"],
            relation_id=row["relation_id"],
            report_id=row["report_id"]
        )

print("Canonical Graph V2 created")
print("Nodes:", G_canonical_v2.number_of_nodes())
print("Edges:", G_canonical_v2.number_of_edges())
print("Density:", nx.density(G_canonical_v2))
print("Weakly connected components:", nx.number_weakly_connected_components(G_canonical_v2))

Canonical Graph V2 created
Nodes: 4363
Edges: 935
Density: 4.9129318138659856e-05
Weakly connected components: 3557


###  Compare Original vs V1 vs V2

In [14]:
comparison_df = pd.DataFrame([
    {
        "graph_version": "Original mention-level graph",
        "nodes": 12679,
        "edges": 1094,
        "density": 0.000007,
        "weakly_connected_components": 11672
    },
    {
        "graph_version": "Canonical graph V1 - normalised exact match",
        "nodes": G_canonical.number_of_nodes(),
        "edges": G_canonical.number_of_edges(),
        "density": nx.density(G_canonical),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical)
    },
    {
        "graph_version": "Canonical graph V2 - noise filtered",
        "nodes": G_canonical_v2.number_of_nodes(),
        "edges": G_canonical_v2.number_of_edges(),
        "density": nx.density(G_canonical_v2),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical_v2)
    }
])

display(comparison_df)

,graph_version,nodes,edges,density,weakly_connected_components
0,Original mention-level graph,12679,1094,0.000007,11672
1,Canonical graph V1 - normalised exact match,4385,945,0.000049,3571
2,Canonical graph V2 - noise filtered,4363,935,0.000049,3557


### Criterion 3


In [15]:
# Criterion 3 – Punctuation-insensitive canonicalisation

def normalise_entity_text_v3(text):
    if pd.isna(text):
        return None

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[-_/\.]", "", text)   # remove punctuation separators
    text = re.sub(r"\s+", " ", text)

    return text

entity_df_v3 = entity_df_clean.copy()
entity_df_v3["normalised_text_v3"] = entity_df_v3["entity_text"].apply(normalise_entity_text_v3)

entity_df_v3["canonical_key_v3"] = (
    entity_df_v3["entity_type"].astype(str)
    + "::"
    + entity_df_v3["normalised_text_v3"].astype(str)
)

print("Unique canonical entities V2:", entity_df_clean["canonical_key"].nunique())
print("Unique canonical entities V3:", entity_df_v3["canonical_key_v3"].nunique())
print("Entities reduced from V2 to V3:", entity_df_clean["canonical_key"].nunique() - entity_df_v3["canonical_key_v3"].nunique())

Unique canonical entities V2: 4363
Unique canonical entities V3: 4354
Entities reduced from V2 to V3: 9


### build V3 graph

In [16]:
id_to_canonical_v3 = dict(
    zip(entity_df_v3["entity_id"], entity_df_v3["canonical_key_v3"])
)

relation_df["canonical_source_v3"] = relation_df["source_id"].map(id_to_canonical_v3)
relation_df["canonical_target_v3"] = relation_df["target_id"].map(id_to_canonical_v3)

canonical_valid_relations_v3 = relation_df[
    relation_df["canonical_source_v3"].notna() &
    relation_df["canonical_target_v3"].notna()
].copy()

G_canonical_v3 = nx.DiGraph()

canonical_entity_df_v3 = (
    entity_df_v3
    .groupby("canonical_key_v3")
    .agg(
        entity_type=("entity_type", "first"),
        canonical_text=("normalised_text_v3", "first"),
        mention_count=("entity_id", "count"),
        original_entity_ids=("entity_id", list),
        report_ids=("report_id", lambda x: list(set(x)))
    )
    .reset_index()
)

for _, row in canonical_entity_df_v3.iterrows():
    G_canonical_v3.add_node(
        row["canonical_key_v3"],
        entity_type=row["entity_type"],
        canonical_text=row["canonical_text"],
        mention_count=row["mention_count"]
    )

for _, row in canonical_valid_relations_v3.iterrows():
    source = row["canonical_source_v3"]
    target = row["canonical_target_v3"]

    if source != target:
        G_canonical_v3.add_edge(
            source,
            target,
            relation_type=row["relation_type"],
            relation_id=row["relation_id"],
            report_id=row["report_id"]
        )

print("Canonical Graph V3 created")
print("Nodes:", G_canonical_v3.number_of_nodes())
print("Edges:", G_canonical_v3.number_of_edges())
print("Density:", nx.density(G_canonical_v3))
print("Weakly connected components:", nx.number_weakly_connected_components(G_canonical_v3))

Canonical Graph V3 created
Nodes: 4354
Edges: 933
Density: 4.922713399625874e-05
Weakly connected components: 3550


### Interpreting the V1 → V2 → V3 Results
Canonicalisation reduced nodes from 12,679 to 4,354 (66%), but weakly connected
components only fell from 11,672 to 3,550 (~30%) — the graph remains highly
fragmented. V2 → V3 merged only 9 additional entities, showing diminishing returns
from further text-normalisation rules. String normalisation alone cannot resolve
most fragmentation: many mentions likely refer to the same entity under different
surface forms (aliases, abbreviations, near-duplicate spellings) that exact
normalisation cannot detect. Criterion 4 below addresses this directly.

### Criterion 4 — Fuzzy Matching (Similarity-Based Canonicalisation)
This criterion merges entities whose normalised text is highly similar but not
identical, using a similarity ratio computed within each entity_type group (never
across types, to avoid semantically nonsensical merges). A conservative threshold
(0.90) is used to minimise false merges.

In [17]:
# SECTION: CRITERION 4 - FUZZY MATCHING WITHIN ENTITY TYPE
import difflib

SIMILARITY_THRESHOLD = 0.90

def build_fuzzy_clusters(df, text_col, type_col, threshold=SIMILARITY_THRESHOLD):
    parent = {}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        parent.setdefault(x, x); parent.setdefault(y, y)
        parent[find(x)] = find(y)

    for entity_type, group in df.groupby(type_col):
        keys = group[["canonical_key_v3", text_col]].drop_duplicates().values.tolist()
        for k, _ in keys:
            parent.setdefault(k, k)
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                key_i, text_i = keys[i]
                key_j, text_j = keys[j]
                if text_i is None or text_j is None:
                    continue
                if difflib.SequenceMatcher(None, text_i, text_j).ratio() >= threshold:
                    union(key_i, key_j)
    return {k: find(k) for k in parent}

fuzzy_cluster_map = build_fuzzy_clusters(entity_df_v3, "normalised_text_v3", "entity_type")
entity_df_v4 = entity_df_v3.copy()
entity_df_v4["canonical_key_v4"] = entity_df_v4["canonical_key_v3"].map(fuzzy_cluster_map)

print("Unique canonical entities V3:", entity_df_v3["canonical_key_v3"].nunique())
print("Unique canonical entities V4:", entity_df_v4["canonical_key_v4"].nunique())
print("Entities reduced from V3 to V4:",
      entity_df_v3["canonical_key_v3"].nunique() - entity_df_v4["canonical_key_v4"].nunique())

Unique canonical entities V3: 4354
Unique canonical entities V4: 4050
Entities reduced from V3 to V4: 304


In [18]:
# SECTION: THRESHOLD SENSITIVITY CHECK
for t in [0.85, 0.90, 0.95]:
    test_map = build_fuzzy_clusters(entity_df_v3, "normalised_text_v3", "entity_type", threshold=t)
    n_unique = len(set(test_map.values()))
    print(f"Threshold {t}: {n_unique} unique canonical entities")

Threshold 0.85: 3882 unique canonical entities
Threshold 0.9: 4050 unique canonical entities
Threshold 0.95: 4257 unique canonical entities


### Build V4 graph

In [19]:
# SECTION: BUILD V4 GRAPH
id_to_canonical_v4 = dict(zip(entity_df_v4["entity_id"], entity_df_v4["canonical_key_v4"]))
relation_df["canonical_source_v4"] = relation_df["source_id"].map(id_to_canonical_v4)
relation_df["canonical_target_v4"] = relation_df["target_id"].map(id_to_canonical_v4)

canonical_valid_relations_v4 = relation_df[
    relation_df["canonical_source_v4"].notna() & relation_df["canonical_target_v4"].notna()
].copy()

G_canonical_v4 = nx.DiGraph()
canonical_entity_df_v4 = (
    entity_df_v4.groupby("canonical_key_v4").agg(
        entity_type=("entity_type", "first"),
        canonical_text=("normalised_text_v3", "first"),
        mention_count=("entity_id", "count"),
        original_entity_ids=("entity_id", list),
        report_ids=("report_id", lambda x: list(set(x)))
    ).reset_index())

for _, row in canonical_entity_df_v4.iterrows():
    G_canonical_v4.add_node(row["canonical_key_v4"], entity_type=row["entity_type"],
                             canonical_text=row["canonical_text"], mention_count=row["mention_count"])

for _, row in canonical_valid_relations_v4.iterrows():
    source, target = row["canonical_source_v4"], row["canonical_target_v4"]
    if source != target:
        G_canonical_v4.add_edge(source, target, relation_type=row["relation_type"],
                                 relation_id=row["relation_id"], report_id=row["report_id"])

print("Canonical Graph V4 created")
print("Nodes:", G_canonical_v4.number_of_nodes())
print("Edges:", G_canonical_v4.number_of_edges())
print("Density:", nx.density(G_canonical_v4))
print("Weakly connected components:", nx.number_weakly_connected_components(G_canonical_v4))

Canonical Graph V4 created
Nodes: 4050
Edges: 910
Density: 5.549304964798502e-05
Weakly connected components: 3267


In [20]:
# SECTION: MANUAL VALIDATION SAMPLE (fuzzy merges specifically)
fuzzy_merged = entity_df_v4[entity_df_v4["canonical_key_v3"] != entity_df_v4["canonical_key_v4"]]
validation_sample = fuzzy_merged.drop_duplicates("canonical_key_v4").sample(
    n=min(20, fuzzy_merged["canonical_key_v4"].nunique()), random_state=42
)
validation_sample.to_csv(OUTPUT_DIR / "canonicalisation_v4_manual_validation_sample.csv", index=False)
display(validation_sample[["entity_type", "normalised_text_v3", "canonical_key_v3", "canonical_key_v4"]])

,entity_type,normalised_text_v3,canonical_key_v3,canonical_key_v4
101,attack-pattern,website defacement,attack-pattern::website defacement,attack-pattern::website defacements
57,TIME,1h’ 2021,TIME::1h’ 2021,TIME::1h 2021
2756,SOFTWARE,alo alto networks nextgeneration firewall with...,SOFTWARE::alo alto networks nextgeneration fir...,SOFTWARE::palo alto networks next generation f...
1798,attack-pattern,system network configuration discovery,attack-pattern::system network configuration d...,attack-pattern::system network configuration d...
1357,FILEPATH,c:\windows\system32\scexe,FILEPATH::c:\windows\system32\scexe,FILEPATH::c:\windows\system32\chexe
746,vulnerability,cve20180802,vulnerability::cve20180802,vulnerability::cve20180807
3827,FILEPATH,silverlightinsexe,FILEPATH::silverlightinsexe,FILEPATH::silverlight2exe
5201,malware,trojanw97mconfuciusb,malware::trojanw97mconfuciusb,malware::trojanw97mconfuciusc
12492,IPV4,174[]139[]62[]60,IPV4::174[]139[]62[]60,IPV4::174[]139[]62[]61
6288,vulnerability,cve202138425,vulnerability::cve202138425,vulnerability::cve202138433


### Compare V1, V2, V3, V4

In [21]:
comparison_df = pd.DataFrame([
    {
        "graph_version": "Original mention-level graph",
        "nodes": 12679,
        "edges": 1094,
        "density": 0.000007,
        "weakly_connected_components": 11672
    },
    {
        "graph_version": "V1 - exact normalised match",
        "nodes": G_canonical.number_of_nodes(),
        "edges": G_canonical.number_of_edges(),
        "density": nx.density(G_canonical),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical)
    },
    {
        "graph_version": "V2 - noise filtered",
        "nodes": G_canonical_v2.number_of_nodes(),
        "edges": G_canonical_v2.number_of_edges(),
        "density": nx.density(G_canonical_v2),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical_v2)
    },
    {
        "graph_version": "V3 - punctuation-insensitive",
        "nodes": G_canonical_v3.number_of_nodes(),
        "edges": G_canonical_v3.number_of_edges(),
        "density": nx.density(G_canonical_v3),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical_v3)
    },
    {
        "graph_version": "V4 - fuzzy matched",
        "nodes": G_canonical_v4.number_of_nodes(),
        "edges": G_canonical_v4.number_of_edges(),
        "density": nx.density(G_canonical_v4),
        "weakly_connected_components": nx.number_weakly_connected_components(G_canonical_v4)
    }
])

display(comparison_df)

,graph_version,nodes,edges,density,weakly_connected_components
0,Original mention-level graph,12679,1094,0.000007,11672
1,V1 - exact normalised match,4385,945,0.000049,3571
2,V2 - noise filtered,4363,935,0.000049,3557
3,V3 - punctuation-insensitive,4354,933,0.000049,3550
4,V4 - fuzzy matched,4050,910,0.000055,3267


### Output

In [22]:
# Save final canonicalisation outputs

# Save V2 graph outputs
with open(OUTPUT_DIR / "cti_knowledge_graph_canonical_v2.pkl", "wb") as file:
    pickle.dump(G_canonical_v2, file)

entity_df_clean.to_csv(OUTPUT_DIR / "canonical_entity_mapping_v2.csv", index=False)
canonical_entity_df_v2.to_csv(OUTPUT_DIR / "kg_canonical_entities_v2.csv", index=False)
canonical_valid_relations_clean.to_csv(OUTPUT_DIR / "kg_canonical_relations_v2.csv", index=False)

# Save V3 graph outputs
with open(OUTPUT_DIR / "cti_knowledge_graph_canonical_v3.pkl", "wb") as file:
    pickle.dump(G_canonical_v3, file)

entity_df_v3.to_csv(OUTPUT_DIR / "canonical_entity_mapping_v3.csv", index=False)
canonical_entity_df_v3.to_csv(OUTPUT_DIR / "kg_canonical_entities_v3.csv", index=False)
canonical_valid_relations_v3.to_csv(OUTPUT_DIR / "kg_canonical_relations_v3.csv", index=False)

# Save V4 (final) graph outputs
with open(OUTPUT_DIR / "cti_knowledge_graph_canonical_final.pkl", "wb") as file:
    pickle.dump(G_canonical_v4, file)

entity_df_v4.to_csv(OUTPUT_DIR / "canonical_entity_mapping_v4.csv", index=False)
canonical_entity_df_v4.to_csv(OUTPUT_DIR / "kg_canonical_entities_v4.csv", index=False)
canonical_valid_relations_v4.to_csv(OUTPUT_DIR / "kg_canonical_relations_v4.csv", index=False)

# Save comparison table (V1-V4)
comparison_df.to_csv(
    OUTPUT_DIR / "graph_original_vs_canonical_v1_v2_v3_v4_comparison.csv",
    index=False
)

# Save merge examples
merge_examples.to_csv(
    OUTPUT_DIR / "canonical_merge_examples.csv",
    index=False
)

print("Canonicalisation V2, V3, and V4 outputs saved successfully.")
print("Final selected graph: cti_knowledge_graph_canonical_final.pkl (V4)")

Canonicalisation V2, V3, and V4 outputs saved successfully.
Final selected graph: cti_knowledge_graph_canonical_final.pkl (V4)


## Conclusion
Four canonicalisation criteria were applied progressively: exact normalised matching
(V1), noise filtering (V2), punctuation-insensitive matching (V3), and similarity-based
fuzzy matching within entity type (V4). Fuzzy matching alone merged 304 additional
entities (4,354 → 4,050), compared to only 9 merged between V2 and V3, showing that
similarity-based matching captured fragmentation that exact text normalisation could
not.

The final canonical graph (V4) contains 4,050 nodes and 910 edges — a 68.1% reduction
in nodes from the original mention-level graph (12,679 nodes, 1,094 edges). Weakly
connected components fell from 11,672 to 3,267 (a 72.0% reduction), and graph density
roughly doubled (from ≈0.000007 to ≈0.0000555). The graph remains fragmented, which is
treated as a stated limitation of working with mention-sparse annotation data rather
than a hidden weakness — see the manual validation sample (Cell 40) for merge-quality
evidence on the fuzzy-matched entities specifically.

The final canonical graph (cti_knowledge_graph_canonical_final.pkl) is the input to
Notebook 5 (embeddings) onward.